<a href="https://colab.research.google.com/github/NikitaVinu/RAG/blob/main/docling_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-qdrant langchain-text-splitters langchain_huggingface langchain-docling langchain-groq

In [2]:
!pip install -q "docling-slim[format-markdown]"

In [3]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

In [5]:
from langchain_docling.loader import DoclingLoader,ExportType
from docling.chunking import HybridChunker
from langchain_core.prompts import PromptTemplate

EXPORT_TYPE = ExportType.DOC_CHUNKS
EMBED_MODEL_ID= "sentence-transformers/all-MiniLM-L6-v2"
WEB_URL = "https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt"

loader = DoclingLoader(
    file_path=WEB_URL,
    export_type=EXPORT_TYPE,
    chunker=HybridChunker(tokenizer=EMBED_MODEL_ID),
)
docs=loader.load()
print(docs)

[Document(metadata={'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}], 'headings': ['AtliqAI HR Policies'], 'origin': {'mimetype': 'text/markdown', 'binary_hash': 12205852925699016857, 'filename': 'atliqai_hr_policies.txt'}}}, page_content='AtliqAI HR Policies\nAtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.'), Document(metadata={'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_po

In [6]:
# Print the metadata of the first Document
print("Metadata of the first document:")
print(docs[0].metadata)

# Print the page content of the first Document
print("\nPage Content of the first document:")
print(docs[0].page_content)


Metadata of the first document:
{'source': 'https://raw.githubusercontent.com/tnahddisttud/sample-doc/refs/heads/main/atliqai_hr_policies.txt', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}], 'headings': ['AtliqAI HR Policies'], 'origin': {'mimetype': 'text/markdown', 'binary_hash': 12205852925699016857, 'filename': 'atliqai_hr_policies.txt'}}}

Page Content of the first document:
AtliqAI HR Policies
AtliqAI is committed to building a transparent, inclusive, and high-performance workplace. This document outlines the policies and guidelines that govern employment, conduct, compensation, and well-being at AtliqAI. All employees are expected to read, understand, and adhere to these policies from their first day of joining.


In [7]:
print(len(docs))

44


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID)

vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embedder,
    path="/tmp/my_lang_vs",
    collection_name="hr_docs",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


In [10]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. Cite section names. Say 'I don't know' if unsure."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

In [11]:
def format_docs(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        dl_meta  = doc.metadata.get("dl_meta", {})
        parts.append(f"[{i}]: \n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

In [12]:
def rag(query: str) -> str:
    docs         = retriever.invoke(query)
    context      = format_docs(docs)
    prompt_value = RAG_PROMPT.invoke({"context": context, "question": query})
    response     = llm.invoke(prompt_value)
    return response.content

In [13]:
for q in [
        "How many casual leaves am I entitled to?",
        "What is the notice period for Band 4 employees?",
        "How long is the probation period?",
    ]:
        print(f"\nQ: {q}\nA: {rag(q)}\n")


Q: How many casual leaves am I entitled to?
A: You are entitled to **12 casual leaves per calendar year** (credited at 1 leave per month)【1†Casual Leave】.


Q: What is the notice period for Band 4 employees?
A: The notice period for Band 4 (and above) employees is **60 days**【1†AtliqAI HR Policies → Separation & Exit → Resignation Process】.


Q: How long is the probation period?
A: The probation period is **6 months** from the date of joining【1:Employment & Onboarding – Probation Period】.



In [14]:
docs_1 = retriever.invoke("How many casual leaves am I entitled to?")

In [15]:
context= format_docs(docs_1)
print(context)

[1]: 
AtliqAI HR Policies
Leave Policy
Casual Leave
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot be carried forward to the next calendar year and lapse on December 31st.

---

[2]: 
AtliqAI HR Policies
Leave Policy
Casual Leave
Every confirmed employee is entitled to 12 casual leaves per calendar year, credited at 1 leave per month. Casual leave can be availed for personal errands, minor illness, or unplanned absences. A maximum of 3 consecutive casual leaves can be taken at a time. Casual leaves cannot be carried forward to the next calendar year and lapse on December 31st.

---

[3]: 
AtliqAI HR Policies
Leave Policy
Sick Leave
Employees are entitled to 10 sick leaves per calendar year. Sick leave can be availed in case of illness, hospitalisation, or 